## **First RAG - From Scratch**

In [16]:
from dotenv import load_dotenv
import os

load_dotenv(".env")
os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")
os.environ["PINECONE_INDEX_NAME"] = os.getenv("PINECONE_INDEX_NAME")

In [3]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

load_dotenv()

loader = PyMuPDFLoader(
    "compressed_Muhammad_Younus_Resume.pdf",
)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8194.41it/s]


In [ ]:
from pinecone import Pinecone, ServerlessSpec
import os

pc = Pinecone(
    api_key="your_pinecone_api_key",
)

index_name = "rag-demo"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("Index ready!")

Index ready!


In [ ]:
pc = Pinecone(
    api_key="your_pinecone_api_key"
)

index = pc.Index(
    "your_pinecone_index_name"
)

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

vector_store.add_documents(chunks)

print("Documents indexed successfully.") 

Documents indexed successfully.


In [23]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3
    }
)

In [24]:
query = "Who is Muhammad Younus and what is his educational background?"

docs = retriever.invoke(query)

In [27]:
for i, doc in enumerate(docs, start=1):
    print(f"\n===== Chunk {i} =====")
    print(doc.page_content)
    print(doc.metadata)


===== Chunk 1 =====
Languages: English · Urdu · Sindhi 
Strengths: Fast Learner · Problem Solver · AI-First Mindset · Self-Motivated
{'author': 'Un-named', 'creationDate': "D:20260602124459+00'00'", 'creationdate': '2026-06-02T12:44:59+00:00', 'creator': 'Microsoft® Word 2016', 'file_path': 'compressed_Muhammad_Younus_Resume.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': 'D:20260606095536Z', 'moddate': '2026-06-06T09:55:36+00:00', 'page': 1.0, 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'source': 'compressed_Muhammad_Younus_Resume.pdf', 'subject': '', 'title': '', 'total_pages': 2.0, 'trapped': ''}

===== Chunk 2 =====
MUHAMMAD YOUNUS 
Agentic AI Developer  ·  Python Engineer  ·  AI Application Builder 
Building production-grade AI systems at age 16 · Karachi, Pakistan 
📧 myounushere@gmail.com 
📧 +92 301 238 1884 
⌨ github.com/MYounus-Codes 
📧 younus.onhercules.app 
📧 LinkedIn Profile 
 
PROFESSIONAL SUMMARY
{'author': 'Un-named', 'creationDate': "D:202606021244

### **Integrating LLM**

In [ ]:
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0
)

##### **Testing LLM**

In [ ]:
### TESTING AN LLM RESPONSE

response = llm.invoke(
    "What is Python?"
)

print(response.content)

<think>
Okay, the user is asking, "What is Python?" I need to provide a clear and concise explanation. Let me start by recalling the basics. Python is a programming language, right? It's known for being easy to read and write. I should mention that it's high-level, which means it's more abstracted from the machine code, making it user-friendly.

Hmm, maybe I should talk about its design philosophy. Guido van Rossum created it in the late 1980s, and it's named after Monty Python, which is a fun fact. The emphasis on code readability with significant whitespace—like using indentation instead of braces. That's a key point because it enforces a clean syntax.

What are the main features? It's interpreted, so code is executed line by line, which is good for debugging. Also, it's dynamically typed, meaning you don't have to declare variable types. Cross-platform compatibility is another plus; Python runs on various operating systems.

Applications of Python are vast. Web development with fram

In [30]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the question ONLY from the provided context.

If the answer is not in the context, say:
"I don't have enough information to answer that."

Context:
{context}

Question:
{question}

Answer:
""")

In [37]:
query = "Who is Muhammad Younus and what is his educational background?"

docs = retriever.invoke(query)

context = "\n\n".join(
    doc.page_content
    for doc in docs
)

messages = prompt.invoke({
    "context": context,
    "question": query
})

response = llm.invoke(messages)

print(response.content)

<think>
Okay, let's see. The user is asking about Muhammad Younus and his educational background. First, I need to check the provided context for any relevant information.

Looking at the context, there's a section that says "Building production-grade AI systems at age 16 · Karachi, Pakistan". This implies that Muhammad was 16 when he was building AI systems. The next line mentions he's a Self-taught Agentic AI Developer and Python Engineer who has shipped production-ready AI systems before completing secondary school. 

So, his educational background would be that he was self-taught and hadn't finished secondary school at the time mentioned. The key points here are self-taught, Agentic AI Developer, Python Engineer, and not having completed secondary school yet. There's no mention of any formal education beyond that. The answer should include his roles and the fact that he was 16 and hadn't finished secondary school. I need to make sure not to include any information not in the contex

## **Interactive ChatBot**

In [ ]:
while True:
    question = input("\nQuestion: ")

    if question.lower() == "exit":
        break

    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    response = llm.invoke(messages)

    print("\nAnswer:")
    print(response.content)